In [ ]:
from torch.utils.data import DataLoader
import random

# Split records before augmentation so equivalent positions do not leak into validation.
shuffled_records = records.copy()
random.Random(0).shuffle(shuffled_records)
train_size = int(0.8 * len(shuffled_records))
train_records = shuffled_records[:train_size]
val_records = shuffled_records[train_size:]

train_dataset = TTTRecordDataset(train_records)
train_dataset.augment_data()
train_dataset = deduplicate_and_average_dataset(train_dataset)

val_dataset = TTTRecordDataset(val_records)
val_dataset = deduplicate_and_average_dataset(val_dataset)

print('Train Size:', len(train_dataset))
print('Val Size:', len(val_dataset))

train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=64, shuffle=False)

criterion_value = nn.MSELoss()

def criterion_policy(logits, target_probs):
    log_probs = torch.log_softmax(logits, dim=1)
    return -(target_probs * log_probs).sum(dim=1).mean()

model = TTTNet()
model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
from torch.utils.data import DataLoader
import random

# Split records before augmentation so equivalent positions do not leak into validation.
shuffled_records = records.copy()
random.Random(0).shuffle(shuffled_records)
train_size = int(0.8 * len(shuffled_records))
train_records = shuffled_records[:train_size]
val_records = shuffled_records[train_size:]

train_dataset = UTTTRecordDataset(train_records)
train_dataset.augment_data()
val_dataset = UTTTRecordDataset(val_records)

print('Train Size:', len(train_dataset))
print('Val Size:', len(val_dataset))

train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=64, shuffle=False)

criterion_value = nn.MSELoss()

def criterion_policy(logits, target_probs):
    log_probs = torch.log_softmax(logits, dim=1)
    return -(target_probs * log_probs).sum(dim=1).mean()

model = UTTTNet()
model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.0005, weight_decay=0.0001)

In [ ]:
# Training loop UTTT
num_epochs = 250
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for states, masks, policies, rewards in train_dataloader:
        states = states.to(device)
        masks = masks.to(device)
        policies = policies.to(device)
        rewards = rewards.to(device)

        # Forward pass
        predicted_policy_logits, predicted_values = model(torch.stack((states, masks), dim=1))

        # Calculate the loss
        policy_loss = criterion_policy(predicted_policy_logits, policies)
        value_loss = criterion_value(predicted_values.squeeze(-1), rewards)
        loss = 0.9 * policy_loss + 0.1 * value_loss

        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * states.size(0)

    avg_train_loss = total_loss / len(train_dataset)

    # Validation loop
    model.eval()
    total_val_loss = 0
    with torch.no_grad():
        for states, masks, policies, rewards in val_dataloader:
            states = states.to(device)
            masks = masks.to(device)
            policies = policies.to(device)
            rewards = rewards.to(device)

            predicted_policy_logits, predicted_values = model(torch.stack((states, masks), dim=1))
            policy_loss = criterion_policy(predicted_policy_logits, policies)
            value_loss = criterion_value(predicted_values.squeeze(-1), rewards)
            loss = 0.9 * policy_loss + 0.1 * value_loss
            total_val_loss += loss.item() * states.size(0)

    avg_val_loss = total_val_loss / len(val_dataset)

    if (epoch + 1) % 10 == 0 or epoch == 0:
      print(f"Epoch [{epoch+1}/{num_epochs}], Train Loss: {avg_train_loss:.4f} Val Loss: {avg_val_loss:.4f}")

print("Training finished.")